### Tools

#### Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [9]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:llama-3.3-70b-versatile")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1112b7bd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1111b9590>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [10]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather at the location"""
    return f"It's sunny is{location}"

In [11]:
model_with_tools=model.bind_tools([get_weather])

In [13]:
# calling the model after binding tools
response=model_with_tools.invoke("what is the weather in Boston")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': '38q46a9wn', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 218, 'total_tokens': 232, 'completion_time': 0.045548855, 'completion_tokens_details': None, 'prompt_time': 0.010952448, 'prompt_tokens_details': None, 'queue_time': 0.161946891, 'total_time': 0.056501303}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ef965-c1ef-75d2-8814-6994be278bef-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '38q46a9wn', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 218, 'output_tokens': 14, 'total_tokens': 232}


### Tools Execution Tools

In [24]:
# step 1 --> model generate tools call
message = [{"role": "user", "content": "what is weather in Boston?"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

# tools call to get result
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

# pass result to ai message
final_response = model_with_tools.invoke(message)

# FIX: changed .text to .content
print(final_response.content) 


The weather in Boston is sunny.


In [25]:
message

[{'role': 'user', 'content': 'what is weather in Boston?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'k5hnhg7t2', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 218, 'total_tokens': 232, 'completion_time': 0.045810459, 'completion_tokens_details': None, 'prompt_time': 0.102605016, 'prompt_tokens_details': None, 'queue_time': 0.41460824, 'total_time': 0.148415475}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef99c-42a5-7110-9b57-e8d814d8787a-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'k5hnhg7t2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 218, 'output_tokens': 14, 'total_tokens': 232}),
 ToolMessage(content="It's sun